**Prérequis : `00_conformite_rgpd_anonymisation.ipynb`** — portique RGPD exécuté et gate GO avant d'ouvrir ce notebook.

---

# TP0 — Ingestion et diagnostic initial (constitution de la couche Bronze)

**Cas d'usage :** prédiction du churn — éditeur SaaS B2B.

**Portée de ce notebook.** Ingérer les fichiers sources exactement tels que reçus du
client, sans aucune transformation, et mener le premier diagnostic (cohérence
structurelle, séparation des cibles, amorce RGPD, recensement des anomalies). **Aucune
correction n'est appliquée ici** — le nettoyage (dédoublonnage, parsing des dates,
typage numérique, normalisation des catégorielles) relève de la couche **Silver**
(notebook suivant).

**Sources brutes** (`Examen_cas d'usage candidat/`) :
- `churn_saas_complet.csv` — dataset principal
- `catalogue_plans.csv` — référentiel des plans tarifaires
- `churn_saas_echantillon.csv` — échantillon fourni en complément

**Sortie produite : la couche Bronze** — copie fidèle des données brutes, tracée et
horodatée, persistée dans `data/bronze/`.

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../Examen_cas d'usage candidat")
BRONZE_DIR = Path("../data/bronze")
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

INGESTED_AT = datetime.now(timezone.utc).isoformat()
LAYER_VERSION = "v1"


def sha256_of(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

print("Répertoire source     :", RAW_DIR.resolve())
print("Répertoire Bronze     :", BRONZE_DIR.resolve())
print("Horodatage d'ingestion:", INGESTED_AT)

Répertoire source     : C:\Users\Aelion\CISIA_UC\Examen_cas d'usage candidat
Répertoire Bronze     : C:\Users\Aelion\CISIA_UC\data\bronze
Horodatage d'ingestion: 2026-09-23T14:43:06.401770+00:00


## §1 — Ingestion brute des fichiers sources

Principe Bronze : **aucune conversion de type, aucune correction**. Tout est lu en
`str` pour ne perdre aucune information — les virgules décimales, les espaces et la
casse d'origine doivent rester visibles pour le diagnostic qui suit. Deux colonnes de
traçabilité (`_source_file`, `_ingested_at_utc`) sont ajoutées ; aucune colonne
métier n'est touchée.

In [2]:
def load_raw(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    df.insert(0, "_source_file", path.name)
    df.insert(1, "_ingested_at_utc", INGESTED_AT)
    return df


clients_raw = load_raw(RAW_DIR / "churn_saas_complet.csv")
catalogue_raw = load_raw(RAW_DIR / "catalogue_plans.csv")
echantillon_raw = load_raw(RAW_DIR / "churn_saas_echantillon.csv")

for name, df in [
    ("clients_raw", clients_raw),
    ("catalogue_raw", catalogue_raw),
    ("echantillon_raw", echantillon_raw),
]:
    print(f"{name:16s}: {df.shape[0]:5d} lignes x {df.shape[1]} colonnes")

clients_raw     :  5035 lignes x 31 colonnes
catalogue_raw   :     4 lignes x 8 colonnes
echantillon_raw :    50 lignes x 31 colonnes


In [3]:
clients_raw.head(5)

,_source_file,_ingested_at_utc,client_id,date_souscription,jour_souscription,secteur,pays,taille_entreprise,plan,anciennete_mois,...,csat,retards_paiement_12m,revenu_mensuel_recurrent_eur,couleur_theme_interface,code_datacenter,groupe_experimentation,commentaire_csm,sante_compte_fin_periode,valeur_vie_client_eur,churn
0,churn_saas_complet.csv,2026-09-23T14:43:06.401770+00:00,CLI-002447,31/01/2024,mercredi,Finance,France,TPE,STARTER,12,...,4,1,"40,0",sombre,us-e1,control,,38,689,0
1,churn_saas_complet.csv,2026-09-23T14:43:06.401770+00:00,CLI-004125,2024-02-06,mardi,SANTÉ,Belgique,TPE,STARTER,12,...,3,0,31.14,violet,us-e1,control,,45,485,0
2,churn_saas_complet.csv,2026-09-23T14:43:06.401770+00:00,CLI-000087,23 Dec 2024,lundi,éducation,France,tpe,Pro,1,...,1,2,171.02,clair,ap-s1,B,Mécontentement exprimé au support.,1,701,1
3,churn_saas_complet.csv,2026-09-23T14:43:06.401770+00:00,CLI-004628,05 Aug 2024,lundi,Santé,,PME,pro,6,...,5,2,771.93,violet,us-e1,control,Faible adoption des sièges.,47,8862,0
4,churn_saas_complet.csv,2026-09-23T14:43:06.401770+00:00,CLI-001671,07 Jan 2025,mardi,Commerce,France,TPE,STARTER,1,...,2,4,"50,01",vert,us-e1,control,,0,300,1


## §2 — Cohérence structurelle

Vérifie que le fichier principal correspond à l'énoncé (colonnes attendues présentes)
et que la clé de jointure `plan` est cohérente avec le référentiel
`catalogue_plans.csv`.

In [4]:
expected_cols = {
    "client_id", "date_souscription", "secteur", "pays", "taille_entreprise", "plan",
    "revenu_mensuel_recurrent_eur", "commentaire_csm", "sante_compte_fin_periode",
    "valeur_vie_client_eur", "churn",
}
missing = expected_cols - set(clients_raw.columns)
print("Colonnes attendues manquantes :", missing or "aucune")
print()

catalogue_plans = sorted(catalogue_raw["plan"])
print("Plans du référentiel catalogue :", catalogue_plans)
print("Valeurs distinctes de 'plan' dans le fichier brut :", clients_raw["plan"].nunique())
print("=> écart attendu : casse et espaces incohérents, à normaliser en Silver,")
print("   pas ici -- la jointure sur catalogue_plans échouerait telle quelle aujourd'hui.")

Colonnes attendues manquantes : aucune

Plans du référentiel catalogue : ['Business', 'Enterprise', 'Pro', 'Starter']
Valeurs distinctes de 'plan' dans le fichier brut : 16
=> écart attendu : casse et espaces incohérents, à normaliser en Silver,
   pas ici -- la jointure sur catalogue_plans échouerait telle quelle aujourd'hui.


## §3 — Séparation des cibles et suspect de fuite (à confirmer plus tard)

Deux cibles distinctes coexistent dans le fichier : `churn` (classification, cible
principale) et `valeur_vie_client_eur` (CLV, cible secondaire). Elles doivent rester
strictement séparées — ni l'une ni l'autre ne doit servir de feature pour prédire
l'autre.

Un diagnostic **à titre indicatif seulement** (conversion en mémoire, rien n'est
modifié dans les données Bronze) permet de repérer une variable suspecte de fuite :
`sante_compte_fin_periode`. Son nom même ("fin de période") suggère un calcul postérieur
à la période observée, donc potentiellement postérieur à la décision de churn.

In [5]:
churn_counts = clients_raw["churn"].value_counts()
print(churn_counts)
taux_churn = churn_counts.get("1", 0) / len(clients_raw)
print(f"Taux de churn brut : {100 * taux_churn:.1f}%")
print()


def to_float_diag(s: str) -> float:
    """Conversion best-effort, pour diagnostic uniquement -- non persistée en Bronze."""
    s = (s or "").strip()
    if s == "":
        return float("nan")
    try:
        return float(s.replace(",", "."))
    except ValueError:
        return float("nan")


sante_diag = clients_raw["sante_compte_fin_periode"].map(to_float_diag)
churn_diag = clients_raw["churn"].astype(float)
corr = sante_diag.corr(churn_diag)
print(f"Corrélation brute (diagnostic) sante_compte_fin_periode / churn : {corr:.3f}")
print("=> Suspect de fuite à investiguer avant modélisation. Aucune décision d'exclusion")
print("   n'est prise ici -- ce n'est qu'un signal à transmettre à l'étape suivante.")

churn
0    3623
1    1412
Name: count, dtype: int64
Taux de churn brut : 28.0%

Corrélation brute (diagnostic) sante_compte_fin_periode / churn : -0.882
=> Suspect de fuite à investiguer avant modélisation. Aucune décision d'exclusion
   n'est prise ici -- ce n'est qu'un signal à transmettre à l'étape suivante.


## §4 — Enjeux éthiques et RGPD (amorce)

Avant tout traitement, un premier passage RGPD sur les données brutes :
- `client_id` : identifiant de compte, pas une donnée nominative.
- `commentaire_csm` : champ texte libre — risque a priori plus élevé, à vérifier.

In [6]:
est_rempli = clients_raw["commentaire_csm"].str.strip() != ""
print(f"Taux de remplissage commentaire_csm : {est_rempli.sum()} / {len(clients_raw)} "
      f"({100 * est_rempli.mean():.1f}%)")
print()
print("Échantillon de valeurs observées :")
for c in clients_raw.loc[est_rempli, "commentaire_csm"].unique()[:8]:
    print(" -", c)

Taux de remplissage commentaire_csm : 2245 / 5035 (44.6%)

Échantillon de valeurs observées :
 - Mécontentement exprimé au support.
 - Faible adoption des sièges.
 - Client très satisfait et actif.
 - Suivi standard.
 - RAS.
 - Client insatisfait, risque de départ.
 - Compte engagé, ambassadeur potentiel.
 - Compte dans la moyenne.


> **Constat.** Sur cet échantillon, le contenu observé est templaté ("Client
> insatisfait, risque de départ.", "Faible adoption des sièges.") — aucun nom
> identifié. **Risque RGPD limité mais pas clos** : une recherche systématique de
> motifs nominatifs (noms, emails, téléphones) sur l'ensemble des valeurs renseignées
> reste à faire avant la couche Silver — volontairement hors périmètre de ce notebook
> Bronze.

## §5 — Anomalies recensées (à traiter en couche Silver, pas ici)

Le principe Bronze interdit toute correction : les anomalies sont **recensées**, pas
corrigées.

In [7]:
cols_metier = [c for c in clients_raw.columns if not c.startswith("_")]
n_doublons = clients_raw.duplicated(subset=cols_metier).sum()

anomalies = pd.DataFrame([
    {"anomalie": "Doublons stricts",
     "detail": f"{n_doublons} lignes strictement dupliquées sur les colonnes métier"},
    {"anomalie": "Formats de date multiples",
     "detail": "au moins 3 formats coexistent dans date_souscription "
               "(ISO, JJ/MM/AAAA, texte anglais 'DD Mon YYYY')"},
    {"anomalie": "Nombres à virgule décimale",
     "detail": "certaines colonnes numériques contiennent des valeurs avec virgule "
               "('33,3') au lieu du point, parfois entre guillemets"},
    {"anomalie": "Pourcentages en texte",
     "detail": "taux_adoption_pct mélange des valeurs suffixées '%' et des valeurs numériques nues"},
    {"anomalie": "Casse / espaces hétérogènes",
     "detail": f"{clients_raw['plan'].nunique()} valeurs distinctes observées pour "
               f"'plan', pour seulement {catalogue_raw['plan'].nunique()} plans réels au catalogue"},
    {"anomalie": "Valeurs manquantes",
     "detail": "commentaire_csm et plusieurs colonnes numériques contiennent des cellules vides"},
])
anomalies

,anomalie,detail
0,Doublons stricts,35 lignes strictement dupliquées sur les colon...
1,Formats de date multiples,au moins 3 formats coexistent dans date_souscr...
2,Nombres à virgule décimale,certaines colonnes numériques contiennent des ...
3,Pourcentages en texte,taux_adoption_pct mélange des valeurs suffixée...
4,Casse / espaces hétérogènes,"16 valeurs distinctes observées pour 'plan', p..."
5,Valeurs manquantes,commentaire_csm et plusieurs colonnes numériqu...


## §6 — Constitution et persistance de la couche Bronze

La couche Bronze est la copie fidèle des fichiers sources, enrichie uniquement des
métadonnées de traçabilité (`_source_file`, `_ingested_at_utc`) — aucune valeur
métier n'est modifiée. Elle est persistée au format Parquet (colonnes `string` pures,
sans inférence de type), accompagnée d'un manifeste JSON.

In [8]:
def write_bronze(df: pd.DataFrame, name: str) -> Path:
    out_path = BRONZE_DIR / f"{name}.parquet"
    df.to_parquet(out_path, index=False)
    return out_path


tables = {
    "clients_churn_bronze": (clients_raw, "churn_saas_complet.csv"),
    "catalogue_plans_bronze": (catalogue_raw, "catalogue_plans.csv"),
    "clients_churn_echantillon_bronze": (echantillon_raw, "churn_saas_echantillon.csv"),
}

manifest = {
    "couche": "bronze",
    "version": LAYER_VERSION,
    "ingested_at_utc": INGESTED_AT,
    "tables": {},
    "avertissement": (
        "Données brutes non corrigées. Aucune transformation (dédoublonnage, parsing "
        "de dates, typage numérique, normalisation catégorielle) n'a été appliquée -- "
        "voir la couche Silver."
    ),
}

for name, (df, source_file) in tables.items():
    out_path = write_bronze(df, name)
    manifest["tables"][name] = {
        "fichier_source": source_file,
        "lignes": int(len(df)),
        "colonnes": list(df.columns),
        "chemin_bronze": str(out_path),
        "sha256_source_csv": sha256_of(RAW_DIR / source_file),
        "sha256_bronze": sha256_of(out_path),
    }

manifest_path = BRONZE_DIR / "bronze_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Couche Bronze écrite dans :", BRONZE_DIR.resolve())
for name, info in manifest["tables"].items():
    print(f" - {info['chemin_bronze']}  ({info['lignes']} lignes)")
print(" -", manifest_path)

Couche Bronze écrite dans : C:\Users\Aelion\CISIA_UC\data\bronze
 - ..\data\bronze\clients_churn_bronze.parquet  (5035 lignes)
 - ..\data\bronze\catalogue_plans_bronze.parquet  (4 lignes)
 - ..\data\bronze\clients_churn_echantillon_bronze.parquet  (50 lignes)
 - ..\data\bronze\bronze_manifest.json


## Journal de bord — Synthèse TP0

- **Ingestion.** 3 fichiers bruts chargés tels quels (`str`, sans inférence de type),
  tracés par `_source_file` / `_ingested_at_utc`.
- **Cohérence structurelle.** Colonnes attendues présentes ; la clé `plan` présente
  des variantes de casse/espace face au catalogue — à normaliser en Silver avant toute
  jointure.
- **Cibles.** `churn` et `valeur_vie_client_eur` restent deux colonnes séparées,
  aucun calcul croisé effectué ici.
- **Suspect de fuite.** `sante_compte_fin_periode` fortement corrélé au churn dès le
  diagnostic brut — à confirmer/exclure avant modélisation (prochaine étape).
- **RGPD (amorce).** `client_id` non nominatif ; `commentaire_csm` rempli à environ
  45%, contenu observé templaté sans nom identifié sur l'échantillon — risque limité
  mais pas clos, recherche systématique à faire en Silver.
- **Anomalies recensées, non corrigées.** Doublons, dates multi-formats, nombres à
  virgule, pourcentages en texte, casse hétérogène.
- **Prochaine étape.** Notebook Silver — nettoyage colonne par colonne, dédoublonnage,
  jointure avec le catalogue, chargement en base relationnelle.